# Ejercicio 3 - Consultas directas sobre archivos Parquet

Exploracion inicial de los viajes de taxis amarillos y verdes de 2026 usando
DuckDB **sin importar los datos a una tabla**: cada consulta lee directamente los
archivos `data/raw/<tipo>/2026/*.parquet` con `read_parquet`,
`parquet_file_metadata` y `parquet_schema`.

Las consultas estan versionadas en `sql/03_exploracion/` (un archivo por
consulta, con su objetivo y fuente en el encabezado). Este notebook las ejecuta
en orden y muestra su resultado. La documentacion completa (consulta, objetivo,
fuente, resultado y decision) esta en `docs/03_consultas_parquet.md`.

Tambien pueden ejecutarse sin el notebook:

```bash
docker exec lab8-lab python scripts/run_sql.py sql/03_exploracion
```

In [1]:
import os
import sys
import time
from pathlib import Path

import pandas as pd

# El notebook corre en notebooks/; las rutas de las consultas son relativas a la raiz.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "scripts"))
from run_sql import conectar, encabezado  # misma configuracion que el script

con = conectar()  # base en memoria, memory_limit = 3GB, cambia a la raiz del proyecto
DIR_SQL = RAIZ / "sql" / "03_exploracion"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)


def ejecutar(nombre: str) -> pd.DataFrame:
    """Muestra el encabezado de sql/03_exploracion/<nombre> y devuelve su resultado."""
    ruta = DIR_SQL / nombre
    print(chr(10).join(encabezado(ruta)))
    inicio = time.perf_counter()
    df = con.sql(ruta.read_text(encoding="utf-8")).df()
    print(f"-- tiempo: {time.perf_counter() - inicio:.2f} s")
    return df


print("DuckDB", con.sql("SELECT version()").fetchone()[0])

DuckDB v1.5.5


## 3.1 Cantidad de archivos disponibles

In [2]:
ejecutar("01_cantidad_archivos.sql")

-- Ejercicio 3.1 - Cantidad de archivos disponibles
-- Objetivo: contar los archivos Parquet descargados por tipo de taxi y anio,
--           junto con su tamanio en disco.
-- Fuente:   data/raw/*/*/*.parquet (todos los archivos descargados)
-- Nota:     parquet_file_metadata() solo lee el pie (footer) de cada archivo,
--           no los datos. GROUP BY ROLLUP agrega una fila de total general.
-- tiempo: 0.09 s


,tipo,anio,archivos,tamanio_mib
0,yellow,2026,8,487.8
1,green,2026,8,7.9
2,TOTAL,,16,495.7


## 3.2 Cantidad de registros

Primero desde los metadatos (sin leer datos) y despues recorriendo los archivos.

In [3]:
ejecutar("02_registros_por_metadatos.sql")

-- Ejercicio 3.2 - Cantidad de registros (desde los metadatos Parquet)
-- Objetivo: obtener la cantidad de filas de cada archivo leyendo solo los
--           metadatos del pie Parquet, sin recorrer los datos.
-- Fuente:   data/raw/*/2026/*.parquet
-- tiempo: 0.03 s


,tipo,mes,filas,row_groups,tamanio_mib,created_by
0,yellow,2026-01,3724889,4,61.2,parquet-cpp-arrow version 16.1.0
1,yellow,2026-02,3399866,4,56.0,parquet-cpp-arrow version 16.1.0
2,yellow,2026-03,3952451,4,64.7,parquet-cpp-arrow version 16.1.0
3,yellow,2026-04,3831240,4,61.8,parquet-cpp-arrow version 24.0.0
4,yellow,2026-05,4090836,4,66.5,parquet-cpp-arrow version 16.1.0
5,yellow,2026-06,3837248,4,62.4,parquet-cpp-arrow version 21.0.0
6,yellow,2026-07,3530109,4,58.8,parquet-cpp-arrow version 21.0.0
7,yellow,2026-08,3336716,4,56.3,parquet-cpp-arrow version 21.0.0
8,green,2026-01,40272,1,0.9,parquet-cpp-arrow version 16.1.0
9,green,2026-02,37373,1,0.9,parquet-cpp-arrow version 16.1.0


In [4]:
ejecutar("03_registros_por_lectura.sql")

-- Ejercicio 3.2 - Cantidad de registros (recorriendo los datos)
-- Objetivo: contar los registros leyendo los archivos con read_parquet y
--           confirmar que coinciden con los metadatos (consulta 02).
-- Fuente:   data/raw/yellow/2026/*.parquet, data/raw/green/2026/*.parquet
-- Nota:     union_by_name = true es necesario porque desde 2026-06 los
--           archivos tienen una columna adicional (request_source).


-- tiempo: 0.36 s


,tipo,archivos,registros
0,yellow,8,29703355.0
1,green,8,337114.0
2,TOTAL,16,30040469.0


## 3.3 Columnas presentes en los archivos

In [5]:
ejecutar("04_columnas.sql")

-- Ejercicio 3.3 - Columnas presentes en los archivos
-- Objetivo: listar las columnas de cada tipo de taxi e indicar en cuantos de
--           los archivos mensuales aparece cada una (detecta cambios de esquema).
-- Fuente:   data/raw/yellow/2026/*.parquet, data/raw/green/2026/*.parquet
-- Nota:     parquet_schema() lee solo los metadatos. La fila raiz 'schema'
--           (num_children no nulo) no es una columna y se excluye.
-- tiempo: 0.08 s


,columna,posicion_yellow,archivos_yellow,posicion_green,archivos_green
0,VendorID,1,8,1,8
1,lpep_pickup_datetime,<NA>,0,2,8
2,tpep_pickup_datetime,2,8,<NA>,0
3,lpep_dropoff_datetime,<NA>,0,3,8
4,tpep_dropoff_datetime,3,8,<NA>,0
5,passenger_count,4,8,8,8
6,trip_distance,5,8,9,8
7,RatecodeID,6,8,5,8
8,store_and_fwd_flag,7,8,4,8
9,PULocationID,8,8,6,8


## 3.4 Tipos de datos

In [6]:
ejecutar("05_tipos_de_datos.sql")

-- Ejercicio 3.4 - Tipos de datos de las columnas
-- Objetivo: obtener, para cada columna, el tipo fisico Parquet, el tipo
--           convertido (logico) y el tipo con el que DuckDB la interpreta, y
--           verificar que el tipo sea el mismo en todos los archivos (si una
--           columna cambiara de tipo apareceria en dos filas).
-- Fuente:   data/raw/yellow/2026/*.parquet, data/raw/green/2026/*.parquet
-- tiempo: 0.03 s


,tipo,columna,tipo_parquet,tipo_convertido,tipo_duckdb,archivos
0,yellow,VendorID,INT32,NaN,INTEGER,8
1,yellow,tpep_pickup_datetime,INT64,TIMESTAMP_MICROS,TIMESTAMP,8
2,yellow,tpep_dropoff_datetime,INT64,TIMESTAMP_MICROS,TIMESTAMP,8
3,yellow,passenger_count,INT64,NaN,BIGINT,8
4,yellow,trip_distance,DOUBLE,NaN,DOUBLE,8
5,yellow,RatecodeID,INT64,NaN,BIGINT,8
6,yellow,store_and_fwd_flag,BYTE_ARRAY,UTF8,VARCHAR,8
7,yellow,PULocationID,INT32,NaN,INTEGER,8
8,yellow,DOLocationID,INT32,NaN,INTEGER,8
9,yellow,payment_type,INT64,NaN,BIGINT,8


## 3.5 Muestra de registros

In [7]:
ejecutar("06_muestra_yellow.sql")

-- Ejercicio 3.5 - Muestra de registros (taxis amarillos)
-- Objetivo: observar registros reales para entender el contenido de cada
--           columna.
-- Fuente:   data/raw/yellow/2026/*.parquet
-- Nota:     muestra pseudoaleatoria de 10 filas tomada de todos los meses: se
--           ordena por el hash de la fila completa. A diferencia de
--           LIMIT 10 (primeras filas del primer archivo) o de
--           USING SAMPLE ... REPEATABLE (que con varios hilos devolvio solo
--           filas del 1 de enero), es reproducible y no esta sesgada al
--           inicio del primer archivo.


-- tiempo: 2.50 s


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,2,2026-07-02 15:31:48,2026-07-02 15:44:54,1,2.57,1,N,142,246,2,15.60,0.0,0.5,0.00,0.0,1.0,20.35,2.5,0.0,0.75,NaN
1,2,2026-07-26 18:20:05,2026-07-26 18:44:09,1,2.99,1,N,68,148,1,21.20,0.0,0.5,5.19,0.0,1.0,31.14,2.5,0.0,0.75,NaN
2,2,2026-07-10 18:08:57,2026-07-10 18:24:43,2,1.34,1,N,90,68,1,14.20,2.5,0.5,4.29,0.0,1.0,25.74,2.5,0.0,0.75,NaN
3,2,2026-06-06 17:40:39,2026-06-06 17:48:19,<NA>,0.93,<NA>,NaN,148,79,0,11.08,0.0,0.5,0.00,0.0,1.0,15.83,NaN,NaN,0.75,HV0003
4,1,2026-08-11 20:46:28,2026-08-11 20:59:10,<NA>,2.40,<NA>,NaN,163,262,0,14.52,0.0,0.5,0.00,0.0,1.0,27.95,NaN,NaN,0.75,HV0003
5,2,2026-02-22 11:13:01,2026-02-22 11:28:00,<NA>,3.63,<NA>,NaN,41,161,0,21.59,0.0,0.5,0.00,0.0,1.0,26.34,NaN,NaN,0.75,NaN
6,2,2026-05-01 14:01:36,2026-05-01 14:35:54,2,7.72,1,N,48,116,1,39.40,0.0,0.5,8.83,0.0,1.0,52.98,2.5,0.0,0.75,NaN
7,2,2026-04-03 12:56:00,2026-04-03 13:17:33,1,1.99,1,N,43,230,1,19.80,0.0,0.5,4.91,0.0,1.0,29.46,2.5,0.0,0.75,NaN
8,2,2026-03-29 22:15:36,2026-03-29 22:23:55,2,2.36,1,N,231,125,1,12.10,1.0,0.5,3.42,0.0,1.0,21.27,2.5,0.0,0.75,NaN
9,2,2026-05-26 17:47:29,2026-05-26 17:49:01,1,0.07,1,N,263,263,4,-3.70,-2.5,-0.5,0.00,0.0,-1.0,-10.20,-2.5,0.0,0.00,NaN


In [8]:
ejecutar("07_muestra_green.sql")

-- Ejercicio 3.5 - Muestra de registros (taxis verdes)
-- Objetivo: observar registros reales para entender el contenido de cada
--           columna.
-- Fuente:   data/raw/green/2026/*.parquet
-- Nota:     ver 06_muestra_yellow.sql.
-- tiempo: 0.10 s


,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,request_source
0,2,2026-06-08 13:23:49,2026-06-08 13:38:54,N,1,7,226,1,2.27,15.60,0.0,0.5,0.00,0.0,NaN,1.0,17.10,2,1,0.00,0.00,NaN
1,2,2026-07-20 19:08:53,2026-07-20 19:22:34,N,1,74,237,1,2.92,16.30,2.5,0.5,1.00,0.0,NaN,1.0,24.05,1,1,2.75,0.00,NaN
2,2,2026-04-01 13:07:15,2026-04-01 13:21:42,N,1,83,70,1,2.03,14.90,0.0,0.5,0.00,0.0,NaN,1.0,16.40,2,1,0.00,0.00,NaN
3,2,2026-08-05 21:37:56,2026-08-05 21:45:24,N,1,82,95,2,1.41,10.00,1.0,0.5,2.50,0.0,NaN,1.0,15.00,1,1,0.00,0.00,NaN
4,2,2026-07-24 20:27:27,2026-07-24 20:32:58,N,1,43,236,1,0.95,7.90,1.0,0.5,1.00,0.0,NaN,1.0,14.15,1,1,2.75,0.00,NaN
5,2,2026-01-04 01:51:57,2026-01-04 02:11:37,N,1,255,211,1,1.90,17.00,1.0,0.5,1.08,0.0,NaN,1.0,24.08,1,1,2.75,0.75,NaN
6,2,2026-08-14 13:53:43,2026-08-14 13:58:59,NaN,<NA>,28,130,<NA>,0.94,6.52,0.0,0.5,0.00,0.0,NaN,1.0,8.02,<NA>,<NA>,NaN,0.00,HV0005
7,2,2026-06-06 10:51:52,2026-06-06 11:04:42,N,1,42,116,1,1.88,12.10,0.0,0.5,0.00,0.0,NaN,1.0,13.60,2,1,0.00,0.00,NaN
8,2,2026-05-10 15:11:50,2026-05-10 15:17:15,N,1,41,166,5,0.77,7.20,0.0,0.5,1.74,0.0,NaN,1.0,10.44,1,1,0.00,0.00,NaN
9,2,2026-02-12 16:25:43,2026-02-12 16:48:39,N,1,95,98,1,5.57,27.50,2.5,0.5,6.30,0.0,NaN,1.0,37.80,1,1,0.00,0.00,NaN


## 3.6 Problemas de calidad de datos

### Perfil estadistico por columna

In [9]:
ejecutar("08_resumen_yellow.sql")

-- Ejercicio 3.6 - Perfil estadistico (taxis amarillos)
-- Objetivo: obtener por columna minimo, maximo, cardinalidad aproximada,
--           promedio, mediana y porcentaje de nulos para detectar valores
--           fuera de rango y columnas incompletas.
-- Fuente:   data/raw/yellow/2026/*.parquet


-- tiempo: 8.09 s


,column_name,column_type,min,max,approx_unique,avg,q50,null_percentage
0,VendorID,INTEGER,1,7,4,1.89,2,0.00
1,tpep_pickup_datetime,TIMESTAMP,2001-01-01 09:23:58,2026-08-31 23:59:59,16867928,NaN,2026-04-30 19:39:06.243608,0.00
2,tpep_dropoff_datetime,TIMESTAMP,2001-01-01 16:09:38,2026-09-01 20:16:00,16587125,NaN,2026-05-01 00:48:54.759864,0.00
3,passenger_count,BIGINT,0,9,11,1.25,1,25.98
4,trip_distance,DOUBLE,0.0,328522.2,7217,5.55,1.8551524541657758,0.00
5,RatecodeID,BIGINT,1,99,7,4.53,1,25.98
6,store_and_fwd_flag,VARCHAR,N,Y,2,NaN,NaN,25.98
7,PULocationID,INTEGER,1,265,290,161.58,161,0.00
8,DOLocationID,INTEGER,1,265,298,161.05,162,0.00
9,payment_type,BIGINT,0,5,6,0.86,1,0.00


In [10]:
ejecutar("09_resumen_green.sql")

-- Ejercicio 3.6 - Perfil estadistico (taxis verdes)
-- Objetivo: igual que 08_resumen_yellow.sql para taxis verdes.
-- Fuente:   data/raw/green/2026/*.parquet


-- tiempo: 0.25 s


,column_name,column_type,min,max,approx_unique,avg,q50,null_percentage
0,VendorID,INTEGER,1,6,3,2.33,2,0.00
1,lpep_pickup_datetime,TIMESTAMP,2008-12-31 17:35:31,2026-08-31 23:58:28,332702,NaN,2026-05-02 12:27:58.868444,0.00
2,lpep_dropoff_datetime,TIMESTAMP,2008-12-31 23:16:26,2026-09-02 09:39:37,396286,NaN,2026-05-02 03:37:58.804573,0.00
3,store_and_fwd_flag,VARCHAR,N,Y,2,NaN,NaN,14.47
4,RatecodeID,BIGINT,1,99,7,1.25,1,14.47
5,PULocationID,INTEGER,1,265,263,97.33,75,0.00
6,DOLocationID,INTEGER,1,265,266,142.88,140,0.00
7,passenger_count,BIGINT,0,9,11,1.30,1,14.47
8,trip_distance,DOUBLE,0.0,179830.92,2472,13.35,2.066346187204347,0.00
9,fare_amount,DOUBLE,-500.0,1676.7,4808,17.01,13.233562665034082,0.00


### Fechas fuera del periodo esperado

In [11]:
ejecutar("10_fechas_fuera_de_rango.sql")

-- Ejercicio 3.6 - Fechas fuera del periodo esperado
-- Objetivo: verificar que la fecha de inicio de cada viaje pertenezca al mes
--           del archivo que lo contiene.
-- Fuente:   data/raw/yellow/2026/*.parquet, data/raw/green/2026/*.parquet
-- Nota:     yellow usa tpep_* y green lpep_*; se unifican como 'inicio'.


-- tiempo: 0.52 s


,tipo,ubicacion,filas,minimo,maximo
0,yellow,1 mes del archivo,29703209,2026-01-01 00:00:00,2026-08-31 23:59:59
1,yellow,2 otro mes de 2026,129,2026-01-31 23:31:23,2026-08-05 20:54:00
2,yellow,3 otro anio,17,2001-01-01 09:23:58,2025-12-31 23:59:06
3,green,1 mes del archivo,337016,2026-01-01 00:03:27,2026-08-31 23:58:28
4,green,2 otro mes de 2026,84,2026-01-26 23:38:06,2026-08-01 14:36:42
5,green,3 otro anio,14,2008-12-31 17:35:31,2025-12-31 22:00:16


### Patron de valores nulos

In [12]:
ejecutar("11_nulos_por_tipo_pago.sql")

-- Ejercicio 3.6 - Patron de valores nulos
-- Objetivo: explicar los nulos que muestra el perfil (08/09) en
--           passenger_count, RatecodeID, store_and_fwd_flag y
--           congestion_surcharge: verificar si ocurren en las mismas filas y
--           si dependen del tipo de pago.
-- Fuente:   data/raw/yellow/2026/*.parquet, data/raw/green/2026/*.parquet
-- Nota:     segun el diccionario de la TLC, payment_type 0 = "Flex Fare trip".


-- tiempo: 0.43 s


,tipo,payment_type,filas,pasajeros_nulos,ratecode_nulos,flag_nulos,congestion_nulos
0,yellow,0,7716688,7716688,7716688,7716688,7716688
1,yellow,1,18941008,0,0,0,0
2,yellow,2,2708031,0,0,0,0
3,yellow,3,98138,0,0,0,0
4,yellow,4,239488,0,0,0,0
5,yellow,5,2,0,0,0,0
6,green,1,219980,0,0,0,0
7,green,2,65921,0,0,0,0
8,green,3,1688,0,0,0,0
9,green,4,750,0,0,0,0


### Valores de las columnas categoricas

In [13]:
ejecutar("12_valores_categoricos.sql")

-- Ejercicio 3.6 - Valores de las columnas categoricas
-- Objetivo: comparar los codigos presentes en las columnas categoricas con
--           los definidos en el diccionario de datos de la TLC para detectar
--           codigos no documentados.
-- Fuente:   data/raw/yellow/2026/*.parquet, data/raw/green/2026/*.parquet
-- Diccionario TLC:
--   VendorID:     1 Creative Mobile Technologies, 2 Curb Mobility,
--                 6 Myle Technologies, 7 Helix
--   RatecodeID:   1 estandar, 2 JFK, 3 Newark, 4 Nassau/Westchester,
--                 5 negociada, 6 viaje grupal, 99 nulo/desconocido
--   payment_type: 0 Flex Fare, 1 tarjeta, 2 efectivo, 3 sin cargo,
--                 4 disputa, 5 desconocido, 6 viaje anulado
--   trip_type:    1 en la calle, 2 despacho (solo green)


-- tiempo: 0.96 s


,columna,valor,yellow,green
0,RatecodeID,1,20102072,269152
1,RatecodeID,2,694936,887
2,RatecodeID,3,90052,203
3,RatecodeID,4,67285,354
4,RatecodeID,5,262614,17739
5,RatecodeID,6,15,2
6,RatecodeID,99,769693,2
7,RatecodeID,NULL,7716688,48775
8,VendorID,1,5467071,28696
9,VendorID,2,23809774,273571


### Reglas de calidad

In [14]:
ejecutar("13_reglas_de_calidad.sql")

-- Ejercicio 3.6 - Reglas de calidad
-- Objetivo: cuantificar registros con valores imposibles o sospechosos segun
--           reglas de negocio simples.
-- Fuente:   data/raw/yellow/2026/*.parquet, data/raw/green/2026/*.parquet
-- Nota:     las reglas NO son excluyentes (una fila puede romper varias).
--           Los LocationID 264 y 265 corresponden a zona desconocida / fuera
--           de NYC en la tabla de zonas de la TLC.


-- tiempo: 1.73 s


,regla,yellow,pct_yellow,green,pct_green
0,01 inicio fuera de 2026,17,0.0001,14,0.0042
1,02 fin antes del inicio,10,0.0000,5,0.0015
2,03 duracion cero,371673,1.2513,229,0.0679
3,04 duracion mayor a 24 h,263,0.0009,4,0.0012
4,05 distancia cero,952231,3.2058,12212,3.6225
5,06 distancia mayor a 100 millas,1223,0.0041,72,0.0214
6,07 tarifa negativa,157364,0.5298,999,0.2963
7,08 total negativo,161835,0.5448,1023,0.3035
8,09 tarifa mayor a 1000 USD,42,0.0001,1,0.0003
9,10 cero pasajeros,91359,0.3076,4527,1.3429


### Registros duplicados

In [15]:
ejecutar("14_duplicados.sql")

-- Ejercicio 3.6 - Registros duplicados
-- Objetivo: detectar filas exactamente repetidas (todas las columnas iguales).
-- Fuente:   data/raw/yellow/2026/*.parquet, data/raw/green/2026/*.parquet
-- Nota:     SELECT DISTINCT * sobre ~30 M filas y 21 columnas agota la memoria
--           del contenedor; en su lugar se compara un hash de 64 bits de la
--           fila completa (hash(t)), que ocupa 8 bytes por fila. La
--           probabilidad de una colision es despreciable (~1e-5 para 30 M
--           filas) y la consulta 15 confirma los duplicados encontrados.


-- tiempo: 2.73 s


,tipo,filas,filas_distintas,duplicados
0,yellow,29703355,29703348,7
1,green,337114,337114,0


In [16]:
ejecutar("15_detalle_duplicados.sql")

-- Ejercicio 3.6 - Detalle de los registros duplicados (taxis amarillos)
-- Objetivo: mostrar las filas repetidas encontradas en la consulta 14 y
--           confirmar que son duplicados reales y no colisiones de hash
--           (count(DISTINCT t) compara la fila completa, columna a columna).
-- Fuente:   data/raw/yellow/2026/*.parquet
-- Nota:     agrupar las ~30 M filas completas agota la memoria; primero se
--           buscan solo los hashes repetidos (8 bytes por fila) y despues se
--           recuperan unicamente esas filas con un SEMI JOIN.


-- tiempo: 5.45 s


,repeticiones,versiones_distintas,VendorID,inicio,fin,origen,destino,distancia,total,payment_type
0,2,1,1,2026-08-20 17:59:03,2026-08-20 17:59:03,264,264,0.0,0.0,4
1,2,1,1,2026-08-20 18:00:55,2026-08-20 18:00:55,264,264,0.0,0.0,4
2,2,1,1,2026-08-20 18:32:01,2026-08-20 18:32:01,264,264,0.0,0.0,4
3,2,1,1,2026-08-20 19:52:55,2026-08-20 19:52:55,264,264,0.0,0.0,4
4,2,1,1,2026-08-20 19:53:55,2026-08-20 19:53:55,264,264,0.0,0.0,4
5,2,1,1,2026-08-22 01:25:29,2026-08-22 01:25:29,264,264,0.0,0.0,4
6,2,1,1,2026-08-23 02:50:26,2026-08-23 02:50:26,264,264,0.0,0.0,4


### Consistencia de `total_amount`

In [17]:
ejecutar("16_consistencia_total.sql")

-- Ejercicio 3.6 - Consistencia de total_amount con sus componentes
-- Objetivo: verificar si total_amount es igual a la suma de los cargos que lo
--           componen y, si no lo es, identificar los patrones de diferencia.
-- Fuente:   data/raw/yellow/2026/*.parquet


-- tiempo: 1.16 s


,payment_type,diferencia,filas,pct_del_total
0,1,0.00,15469459,52.08
1,0,2.50,4747991,15.98
2,1,-3.25,2440215,8.22
3,2,0.00,2269576,7.64
4,0,0.00,812122,2.73
5,1,-2.50,646881,2.18
6,2,-3.25,294687,0.99
7,4,0.00,209877,0.71
8,0,5.50,112169,0.38
9,0,3.50,86542,0.29


## 3.9 Evidencia: por que consultar el Parquet directamente

Tres experimentos pequenos que respaldan la explicacion del documento:

1. **Metadatos**: contar filas no requiere leer datos. `parquet_file_metadata` lee solo
   el pie de cada archivo, y DuckDB responde `count(*)` sin filtros con los conteos de
   los row groups, por lo que ambos tardan fracciones de segundo. Una agregacion sobre
   una columna (`sum(total_amount)`) si debe descomprimir esa columna, pero solo esa.
2. **Projection y filter pushdown**: el plan muestra que DuckDB solo lee las columnas
   necesarias y aplica los filtros dentro del lector Parquet.
3. **DuckDB vs. cargar en memoria con pandas**.

In [18]:
def medir(sql: str):
    inicio = time.perf_counter()
    resultado = con.sql(sql).fetchall()
    return resultado, round(time.perf_counter() - inicio, 3)

por_metadatos = medir("SELECT sum(num_rows) FROM parquet_file_metadata('data/raw/*/2026/*.parquet')")
por_conteo = medir("SELECT count(*) FROM read_parquet('data/raw/*/2026/*.parquet', union_by_name = true)")
por_columna = medir("SELECT round(sum(total_amount)) FROM read_parquet('data/raw/*/2026/*.parquet', union_by_name = true)")
pd.DataFrame(
    [("parquet_file_metadata: sum(num_rows)", *por_metadatos),
     ("read_parquet: count(*)", *por_conteo),
     ("read_parquet: sum(total_amount) (lee 1 de 22 columnas)", *por_columna)],
    columns=["estrategia", "resultado", "segundos"],
)

,estrategia,resultado,segundos
0,parquet_file_metadata: sum(num_rows),"[(30040469,)]",0.071
1,read_parquet: count(*),"[(30040469,)]",0.191
2,read_parquet: sum(total_amount) (lee 1 de 22 c...,"[(901765045.0,)]",0.434


In [19]:
plan = con.sql("""
    EXPLAIN
    SELECT avg(trip_distance)
    FROM read_parquet('data/raw/yellow/2026/*.parquet', union_by_name = true)
    WHERE tpep_pickup_datetime >= '2026-07-01' AND payment_type = 2
""").fetchall()[0][1]
print(plan)

┌───────────────────────────┐
│    UNGROUPED_AGGREGATE    │
│    ────────────────────   │
│    Aggregates: avg(#0)    │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│       trip_distance       │
│                           │
│      ~5,940,671 rows      │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│        READ_PARQUET       │
│    ────────────────────   │
│         Function:         │
│        READ_PARQUET       │
│                           │
│        Projections:       │
│       trip_distance       │
│                           │
│          Filters:         │
│   tpep_pickup_datetime>=  │
│   '2026-07-01 00:00:00':  │
│         :TIMESTAMP        │
│       payment_type=2      │
│                           │
│      ~5,940,671 rows      │
└───────────────────────────┘



In [20]:
import json
import subprocess

# Cada estrategia corre en un proceso Python nuevo para que la memoria maxima
# no incluya lo usado por las consultas anteriores del notebook. Se usa VmHWM
# (pico de memoria residente del proceso, Linux), que se reinicia en cada
# proceso nuevo; ru_maxrss en cambio se hereda del proceso padre.
MEDIR = """
import json, time
inicio = time.perf_counter()
{codigo}
segundos = time.perf_counter() - inicio
vmhwm_kib = next(int(l.split()[1]) for l in open('/proc/self/status') if l.startswith('VmHWM'))
print(json.dumps({{"resultado": round(resultado, 3), "segundos": round(segundos, 2),
                  "memoria_max_mib": round(vmhwm_kib / 1024)}}))
"""

estrategias = {
    "DuckDB: avg sobre los 8 meses (29.7 M filas), directo del Parquet": """
import duckdb
resultado = duckdb.sql("SELECT avg(trip_distance) FROM read_parquet('data/raw/yellow/2026/*.parquet', union_by_name = true)").fetchone()[0]
""",
    "pandas: cargar 1 mes (3.7 M filas) completo en memoria y promediar": """
import pandas as pd
resultado = pd.read_parquet('data/raw/yellow/2026/yellow_tripdata_2026-01.parquet')['trip_distance'].mean()
""",
}

filas = []
for nombre, codigo in estrategias.items():
    salida = subprocess.run([sys.executable, "-c", MEDIR.format(codigo=codigo)],
                            cwd=RAIZ, capture_output=True, text=True, check=True).stdout
    filas.append({"estrategia": nombre, **json.loads(salida)})
pd.DataFrame(filas)

,estrategia,resultado,segundos,memoria_max_mib
0,"DuckDB: avg sobre los 8 meses (29.7 M filas), ...",5.553,0.39,93
1,pandas: cargar 1 mes (3.7 M filas) completo en...,6.456,0.91,1437
